In [413]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import scipy as sp
%matplotlib qt
from continuum_solver import ContinuumSolver, TimeDependentSolver, clean_input, matrix_exp
%load_ext autoreload
%autoreload 2

torch.backends.cuda.preferred_linalg_library("magma")
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device is {DEVICE}")
# set default datatype of tensors
DTYPE = torch.complex128

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Device is cuda


In [414]:
def harmonic_oscillator(x, x_max=5):
    return (1/2)*(x - x_max/2)**2

def mirror_cavity(x, g_0, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    a_mirror = 2 * np.cos(dx) + g_0 * np.sin(dx)
    r = (a_mirror - np.sqrt(a_mirror**2 -  4)) / 2
    g_defect = (2 * r - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def cavity(x, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    g_0 = 2 * (2 - np.cos(dx)) / np.sin(dx)
    g_defect = (2 * (2 - np.sqrt(3)) - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def potential_fourier(x, x_min, x_max, x_steps, f_steps_min):
    V = torch.zeros((x_steps, 2 * f_steps_min + 1), device=DEVICE, dtype=DTYPE)
    V[:, f_steps_min] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))
    V[:, f_steps_min + 1] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))
    V[:, f_steps_min - 1] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))

    return V

def fourier_harmonic_oscillator(x, x_min, x_max, x_steps, f_steps_min):
    V = torch.zeros((x_steps, 2 * f_steps_min + 1), device=DEVICE, dtype=DTYPE)
    V[:, f_steps_min] = 0.5 * (x - (x_max - x_min) / 2)**2

    return V

In [415]:
x_max = 4
x_steps = 4096
f_steps_min = 0
epsilon = 2e-10
k_vals = torch.linspace(0, 2*torch.pi, 3, device=DEVICE)
energies = torch.linspace(0, 5, 500, device=DEVICE, dtype=DTYPE)

solver_ho = ContinuumSolver(
    lambda x: harmonic_oscillator(x, x_max=x_max),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_mirror = ContinuumSolver(
    lambda x: cavity(x, 0, x_max, x_steps),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_time_dep = TimeDependentSolver(
    lambda x, f: fourier_harmonic_oscillator(
        x, x_min=0, x_max=x_max, x_steps=x_steps, f_steps_min=f
    ),
    x_min=0.,
    x_max=x_max,
    x_steps=x_steps,
    f_steps_min=f_steps_min,
    omega=1
)

In [416]:
loss_t_dep = solver_time_dep.loss(energies, 0)
loss_ho = solver_ho.loss(0, energies)

In [417]:
(loss_t_dep - loss_ho).mean()

tensor(-4.6095e-14+0.j, device='cuda:0', dtype=torch.complex128)

In [418]:
# solver_time_dep.plot_loss(energies, 0)

In [419]:
# solver_ho.plot_loss(0, energies)

In [420]:
fig, ax = plt.subplots(figsize=(10, 8))

E_cpu = energies.cpu()

ax.plot(E_cpu, loss_t_dep.cpu(), label="Time-Dependent Loss")
ax.plot(E_cpu, loss_ho.cpu(), label="Old Loss")

ax.legend()
ax.grid()
ax.set_title("Losses vs. Energy Plot")

ax.set_xlabel("Energy")
ax.set_ylabel("Loss")

plt.show()

In [421]:
# eye = torch.eye(solver_time_dep.f_steps, device=DEVICE, dtype=DTYPE)
# fourier_coeffs = solver_time_dep.V(solver_time_dep.x_vals, solver_time_dep.f_steps_min)[:, solver_time_dep.f_steps_min]
# V_shape = solver_time_dep.x_vals.shape + solver_time_dep.f_vals.shape